In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ==========================================
# 1. CONFIG
# ==========================================
DATA_DIR = "/kaggle/input/competitions/datathon-2026-round-1/"

# ==========================================
# 2. LOAD DATA
# ==========================================
sales = pd.read_csv(DATA_DIR + "sales.csv", parse_dates=["Date"])
web = pd.read_csv(DATA_DIR + "web_traffic.csv", parse_dates=["date"])
sample_sub = pd.read_csv(DATA_DIR + "sample_submission.csv", parse_dates=["Date"])

# ==========================================
# 3. FEATURE ENGINEERING (NO EXTERNAL DATA)
# ==========================================
def add_time_features(df):
    df = df.copy()

    df["month"] = df.Date.dt.month
    df["dayofweek"] = df.Date.dt.dayofweek
    df["dayofyear"] = df.Date.dt.dayofyear
    df["weekofyear"] = df.Date.dt.isocalendar().week.astype(int)

    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

    # Cyclical encoding (🔥 rất mạnh)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

    df["dow_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)

    df["doy_sin"] = np.sin(2 * np.pi * df["dayofyear"] / 365)
    df["doy_cos"] = np.cos(2 * np.pi * df["dayofyear"] / 365)

    # Proxy Tết (🔥 hợp lệ vì derive từ Date)
    df["is_tet_proxy"] = (
        ((df["month"] == 1) & (df["dayofyear"] < 40)) |
        ((df["month"] == 2) & (df["dayofyear"] < 60))
    ).astype(int)

    return df


# ==========================================
# 4. LAG + ROLLING
# ==========================================
def add_lag_features(df):
    df = df.sort_values("Date").copy()

    for lag in [7, 14, 30, 90, 365]:
        df[f"lag_{lag}"] = df["Revenue"].shift(lag)

    for w in [7, 30, 90]:\
        df[f"roll_mean_{w}"] = df["Revenue"].shift(1).rolling(w).mean()
        df[f"roll_std_{w}"] = df["Revenue"].shift(1).rolling(w).std()

    return df


# ==========================================
# 5. WEB TRAFFIC (ANTI-LEAKAGE)
# ==========================================
web_daily = web.groupby("date").agg({
    "sessions": "sum",
    "bounce_rate": "mean"
}).reset_index().rename(columns={"date": "Date"})

web_daily["sessions_lag_1"] = web_daily["sessions"].shift(1)
web_daily["bounce_lag_1"] = web_daily["bounce_rate"].shift(1)

# ==========================================
# 6. MERGE DATA
# ==========================================
full_df = pd.concat([sales, sample_sub], axis=0).sort_values("Date")

full_df = add_lag_features(full_df)
full_df = add_time_features(full_df)

full_df = pd.merge(
    full_df,
    web_daily[["Date", "sessions_lag_1", "bounce_lag_1"]],
    on="Date",
    how="left"
)

# ==========================================
# 7. SPLIT
# ==========================================
train_full = full_df[full_df["Date"] <= "2022-12-31"].dropna()
test_part = full_df[full_df["Date"] >= "2023-01-01"]

features = [col for col in train_full.columns 
            if col not in ["Date", "Revenue", "COGS", "sessions", "bounce_rate"]]

X = train_full[features]
y = np.log1p(train_full["Revenue"])

split_date = "2022-01-01"

X_train = X[train_full["Date"] < split_date]
y_train = y[train_full["Date"] < split_date]

X_val = X[train_full["Date"] >= split_date]
y_val = train_full.loc[train_full["Date"] >= split_date, "Revenue"]

# ==========================================
# 8. MODEL
# ==========================================
params = {
    "objective": "regression_l1",
    "learning_rate": 0.03,
    "num_leaves": 63,
    "feature_fraction": 0.8,
    "subsample": 0.8,
    "random_state": 42,
    "verbosity": -1
}

model = lgb.LGBMRegressor(**params, n_estimators=2000)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, np.log1p(y_val))],
    callbacks=[lgb.early_stopping(100)]
)

# ==========================================
# 9. EVALUATION
# ==========================================
val_preds_log = model.predict(X_val)
val_preds = np.expm1(val_preds_log)

mae = mean_absolute_error(y_val, val_preds)
rmse = np.sqrt(mean_squared_error(y_val, val_preds))
r2 = r2_score(y_val, val_preds)

mape = np.mean(np.abs((y_val - val_preds) / np.clip(y_val, 1, None))) * 100

smape = np.mean(
    np.abs(y_val - val_preds) / ((np.abs(y_val) + np.abs(val_preds)) / 2)
) * 100

print("===== MODEL PERFORMANCE =====")
print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R2   : {r2:.4f}")
print(f"MAPE : {mape:.2f}%")
print(f"SMAPE: {smape:.2f}%")

# ==========================================
# 10. SHAP
# ==========================================
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_val)
shap.summary_plot(shap_values, X_val, plot_type="bar")

# ==========================================
# 11. COGS
# ==========================================
train_full["cogs_ratio"] = train_full["COGS"] / train_full["Revenue"].clip(lower=1)
current_cogs_ratio = train_full["cogs_ratio"].tail(30).mean()

# ==========================================
# 12. PREDICT
# ==========================================
test_preds_log = model.predict(test_part[features])
test_revenue = np.expm1(test_preds_log)

submission = pd.DataFrame({
    "Date": test_part["Date"].dt.strftime("%Y-%m-%d"),
    "Revenue": test_revenue.round(2),
    "COGS": (test_revenue * current_cogs_ratio).round(2)
})

submission.to_csv("submission_final.csv", index=False)

print("✅ DONE!")


Error: No connection selected.